# CAMELS: plot
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 15-04-2026*<br>

**Introducción:**<br>


**Outputs:**<br>


**To do**:<br>


In [1]:
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm
from pathlib import Path

from ocab.config import Config
from ocab.plots import plot_station_timeseries, create_station_html

In [ ]:
import unicodedata
import re

def slugify(value):
    # Remove accents
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    # Remove non-alphanumeric characters and replace spaces with hyphens
    value = re.sub(r'[^\w\s-]', '', value).strip().lower()
    return re.sub(r'[-\s]+', '-', value)

## Configuration

In [5]:
cfg = Config('config_CAMELS_v200.yml')

# path time series
path_timeseries = Path('../../docs/timeseries/stations')
path_out = path_timeseries / 'plots'
path_out.mkdir(parents=False, exist_ok=True)

## Data

### Outlets


In [6]:
# load basin outlets
outlets = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')
print(f'{len(outlets)} basin outlets')

901 basin outlets


### Time series

In [11]:
for ID in tqdm(outlets.index):
    # extract attributes and time series
    attrs = outlets.loc[ID]
    ts = pd.read_parquet(path_timeseries / f'{ID}.parquet')
    
    # create time series plot
    title = '{0} - {1} - River {2} ({3})'.format(
        ID, 
        attrs['name'].title(), 
        attrs['river'].title(), 
        attrs['basin'].title()
    )
    fig = plot_station_timeseries(
        ts,
        area=attrs['catch_skm'],
        title=title,
        regime=attrs['regime'],
        save=True
    )

    # save plot as HTML
    create_station_html(
        fig, 
        path=path_out / f'{ID}.html', 
        start=ts.index.min().strftime('%Y-%m-%d'), 
        end=ts.index.max().strftime('%Y-%m-%d')
    )


  0%|          | 0/901 [00:00<?, ?it/s]